In [8]:
import importlib.util

V3_DATASET_PATH = (
    PROJECT_ROOT
    / "src"
    / "v3_2.5d_unet_model"
    / "dataset.py"
)

spec = importlib.util.spec_from_file_location(
    "v3_dataset",
    V3_DATASET_PATH
)

v3_dataset_module = (
    importlib.util.module_from_spec(spec)
)

spec.loader.exec_module(
    v3_dataset_module
)

BraTS25DDataset = (
    v3_dataset_module.BraTS25DDataset
)

print(
    "V3 dataset class loaded successfully"
)

V3 dataset class loaded successfully


In [9]:
# ==================================================
# Load V3 Dataset
# ==================================================

TRAIN_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "v3"
    / "train"
)

VAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "v3"
    / "val"
)

train_dataset = BraTS25DDataset(
    TRAIN_DIR
)

val_dataset = BraTS25DDataset(
    VAL_DIR
)

print()
print(
    "Train samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(val_dataset)
)

Loaded 999 patients with 16000 samples
Loaded 252 patients with 4000 samples

Train samples: 16000
Validation samples: 4000


In [10]:
image, mask = train_dataset[0]

print("Input shape:", image.shape)
print("Target shape:", mask.shape)

print("Input dtype:", image.dtype)
print("Target dtype:", mask.dtype)

print("Target values:", torch.unique(mask))

Input shape: torch.Size([12, 128, 128])
Target shape: torch.Size([128, 128])
Input dtype: torch.float32
Target dtype: torch.float32
Target values: tensor([0., 1.])


In [11]:
# ==================================================
# V3 DataLoader
# ==================================================

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

images, masks = next(
    iter(train_loader)
)

print("Batch images:", images.shape)
print("Batch masks:", masks.shape)

print("Image dtype:", images.dtype)
print("Mask dtype:", masks.dtype)

Batch images: torch.Size([8, 12, 128, 128])
Batch masks: torch.Size([8, 128, 128])
Image dtype: torch.float32
Mask dtype: torch.float32


In [14]:
# ==================================================
# Device + V3 DataLoader Benchmark
# ==================================================

import time
import torch

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

print("\nRunning 100 V3 batches...")

start = time.time()

for i, (images, masks) in enumerate(train_loader):

    images = images.to(device)
    masks = masks.to(device)

    if i == 99:
        break

elapsed = time.time() - start

print()
print("Batches:", 100)
print("Total time:", round(elapsed, 2), "seconds")
print(
    "Time/batch:",
    round(elapsed / 100, 3),
    "seconds"
)

batches_per_epoch = len(train_loader)

estimated_epoch = (
    (elapsed/100)
    * batches_per_epoch
    / 60
)

print(
    "Batches/epoch:",
    batches_per_epoch
)

print(
    "Estimated data-loading epoch time:",
    round(estimated_epoch, 2),
    "minutes"
)

Device: mps

Running 100 V3 batches...

Batches: 100
Total time: 6.61 seconds
Time/batch: 0.066 seconds
Batches/epoch: 2000
Estimated data-loading epoch time: 2.2 minutes


In [15]:
from pathlib import Path

V2_UNET = (
    PROJECT_ROOT
    / "src"
    / "v2_2d_unet_model"
    / "unet.py"
)

print(V2_UNET)
print(V2_UNET.exists())

print("\n--- V2 U-Net ---\n")

print(V2_UNET.read_text())

/Users/abhra/Downloads/AI-ML/Resume/BraTS/src/v2_2d_unet_model/unet.py
True

--- V2 U-Net ---

import torch
import torch.nn as nn


class DoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):
        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),
        )

    def forward(self, x):

        re

In [16]:
# ==================================================
# Load V3 Model
# ==================================================

import importlib.util

V3_MODEL_PATH = (
    PROJECT_ROOT
    / "src"
    / "v3_2.5d_unet_model"
    / "unet_v3.py"
)

spec = importlib.util.spec_from_file_location(
    "unet_v3",
    V3_MODEL_PATH
)

v3_model_module = (
    importlib.util.module_from_spec(spec)
)

spec.loader.exec_module(
    v3_model_module
)

UNetV3 = v3_model_module.UNetV3

model = UNetV3().to(device)

print(model)

UNetV3(
  (enc1): DoubleConv(
    (block): Sequential(
      (0): Conv2d(12, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc2): DoubleConv(
    (block): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU(inplace=True)

In [17]:
# ==================================================
# Forward Pass Test
# ==================================================

images, masks = next(
    iter(train_loader)
)

images = images.to(device)
masks = masks.to(device)

print("Input:", images.shape)

with torch.no_grad():

    logits = model(images)

print("Output:", logits.shape)

Input: torch.Size([8, 12, 128, 128])
Output: torch.Size([8, 1, 128, 128])


In [18]:
# ==================================================
# Forward + Backward Test
# ==================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

criterion = torch.nn.BCEWithLogitsLoss()

model.train()

optimizer.zero_grad()

logits = model(images)

loss = criterion(
    logits,
    masks.unsqueeze(1)
)

loss.backward()

optimizer.step()

print(
    "Loss:",
    loss.item()
)

print(
    "Backward pass successful"
)

Loss: 0.6290410757064819
Backward pass successful


In [19]:
# ==================================================
# V3 Training Benchmark
# ==================================================

import time
import torch

model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

criterion = torch.nn.BCEWithLogitsLoss()

print("Running 100 V3 training batches...")

start = time.time()

for i, (images, masks) in enumerate(train_loader):

    images = images.to(device)
    masks = masks.to(device).unsqueeze(1)

    optimizer.zero_grad()

    logits = model(images)

    loss = criterion(
        logits,
        masks
    )

    loss.backward()

    optimizer.step()

    if i == 99:
        break

elapsed = time.time() - start

time_per_batch = elapsed / 100
batches_per_epoch = len(train_loader)

estimated_epoch = (
    time_per_batch
    * batches_per_epoch
    / 60
)

print()
print("Batches:", 100)
print(
    "Total time:",
    round(elapsed, 2),
    "seconds"
)

print(
    "Time/batch:",
    round(time_per_batch, 3),
    "seconds"
)

print(
    "Batches/epoch:",
    batches_per_epoch
)

print(
    "Estimated epoch time:",
    round(estimated_epoch, 2),
    "minutes"
)

Running 100 V3 training batches...

Batches: 100
Total time: 20.69 seconds
Time/batch: 0.207 seconds
Batches/epoch: 2000
Estimated epoch time: 6.9 minutes


In [20]:
from pathlib import Path

V2_DIR = (
    PROJECT_ROOT
    / "src"
    / "v2_2d_unet_model"
)

print("V2 files:")

for path in sorted(V2_DIR.glob("*.py")):
    print(path.name)

V2 files:
benchmark_training.py
build_v2_metadata.py
build_v2_sample.py
build_v2_slice_index.py
dataset.py
preprocess_v2.py
test_dataset.py
test_preprocess.py
test_training.py
test_unet.py
train_v2.py
unet.py


In [21]:
V2_TRAIN = (
    PROJECT_ROOT
    / "src"
    / "v2_2d_unet_model"
    / "train_v2.py"
)

print(V2_TRAIN.read_text())

from pathlib import Path
import sys
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader


# ==================================================
# Paths
# ==================================================

PROJECT_ROOT = Path(__file__).resolve().parents[1]

sys.path.append(
    str(PROJECT_ROOT / "src")
)

from v2_2d_unet_model.dataset import BraTSSliceDataset
from v2_2d_unet_model.unet import UNet


# ==================================================
# Configuration
# ==================================================

BATCH_SIZE = 8
NUM_WORKERS = 0

EPOCHS = 10
LEARNING_RATE = 1e-4

THRESHOLD = 0.5

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "models"
    / "v2"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================
# Device
# ==================================================

if torch.backends.mps.is_available():

    device = torch.device("mps")

else:

    device = torch.device("cpu")